In [1]:
import accelforge as af

def trunc(s):
    return s.rsplit("/", 1)[-1].rsplit(".yaml", 1)[0]


def map(arch, workload):
    spec = af.Spec.from_yaml(
        arch,
        workload
    )
    spec.mapper.metrics = af.Metrics.LATENCY | af.Metrics.ENERGY 
    mapping = spec.map_workload_to_arch()

    filename = trunc(arch) + "-" + trunc(workload)
    with open("images/" + filename + ".svg", "w") as f:
        f.write(mapping.render())

    actions = mapping.actions(per_einsum=True, per_component=True)
    latency = mapping.latency(per_einsum=True)
    
    total_reads = 0
    total_writes = 0
    for (einsum, lat) in latency.items():
        component = "MainMemory"
        reads = actions[(einsum, component, "read")]
        writes = actions[(einsum, component, "write")]
        total_reads += reads
        total_writes += writes

    rw = total_reads + total_writes
    latency = mapping.latency()

    mem_actions = {}
    for (op, mem, action), value in actions.items():
        if action != "compute" and mem == "MainMemory":   # ignore compute entries
            key = (op, mem)
            mem_actions[key] = mem_actions.get(key, 0.0) + value
    
    return {
        'mapping': mapping,
        'total reads': total_reads,
        'total write': total_writes,
        'total r+w': rw,
        'latency': latency,
        '(r+w)/latency': rw / latency,
        'bw util': (rw/latency) / 6.4e9, # todo ugly i am hardcoded
        'mem summary': mem_actions
    }


In [5]:
%%capture captured_output 
# view stdout by running captured_output.show()

gpt_full = map(
    "eyeriss-submods/full-boundmainmemlat.yaml",
    "workload/gpt3_6.7B-uniquenames.yaml"
)

FileNotFoundError: Could not find file eyeriss-submods/full-boundmainmemlat.yaml

In [1]:
%%capture captured_output 
# view stdout by running captured_output.show()

mmv_full = map(
    "eyeriss-submods/full.yaml",
    "matmulvecmul/full-workload.yaml"
)

In [147]:
full_mapping = mmv_full

In [148]:
with open("image.svg", "w") as f:
    f.write(full_mapping['mapping'].render())

In [137]:
full_mapping

{'mapping': <accelforge.mapper.FFM.mappings.Mappings at 0x7f6c46e421e0>,
 'total reads': 206426865664.0,
 'total write': 37044092928.0,
 'total r+w': 243470958592.0,
 'latency': np.float64(182.70388112962246),
 '(r+w)/latency': np.float64(1332598722.5157263),
 'bw util': np.float64(0.20821855039308224),
 'mem summary': {('I', 'MainMemory'): 0.0,
  ('V', 'MainMemory'): 13153337344.0,
  ('K', 'MainMemory'): 13153337344.0,
  ('Q', 'MainMemory'): 13153337344.0,
  ('QK', 'MainMemory'): 26038239232.0,
  ('QK_softmax', 'MainMemory'): 34359738368.0,
  ('AV', 'MainMemory'): 26038239232.0,
  ('Z', 'MainMemory'): 13153337344.0,
  ('FFA', 'MainMemory'): 52613349376.0,
  ('FFB', 'MainMemory'): 51808043008.0}}

In [138]:
full_mapping['mapping'].actions()

{('MainMemory', 'read'): 206426865664.0,
 ('MainMemory', 'write'): 37044092928.0,
 ('FastMAC', 'compute'): 2201170739200.0,
 ('FastInputScratchpad', 'read'): 17609365913600.0,
 ('FastInputScratchpad', 'write'): 146028888064.0,
 ('FastOutputScratchpad', 'read'): 21848498634752.0,
 ('FastOutputScratchpad', 'write'): 21848498634752.0,
 ('FastGlobalBuffer', 'read'): 17179869184.0,
 ('FastGlobalBuffer', 'write'): 9533652992.0,
 ('FastWeightScratchpad', 'read'): 17592186044416.0,
 ('FastWeightScratchpad', 'write'): 4398046511104.0}

In [139]:
full_mapping['mapping'].latency(per_component=True, per_einsum=True)

{('I', 'MainMemory'): 0.0,
 ('I', 'FastMAC'): 0.16777215898036957,
 ('V', 'FastMAC'): 10.737418174743652,
 ('V', 'FastInputScratchpad'): 0.80530846118927,
 ('V', 'MainMemory'): 2.055208921432495,
 ('V', 'FastOutputScratchpad'): 1.9945430755615234,
 ('V', 'FastGlobalBuffer'): 0.26214399933815,
 ('V', 'FastWeightScratchpad'): 0.322990357875824,
 ('K', 'FastMAC'): 10.737418174743652,
 ('K', 'FastInputScratchpad'): 0.80530846118927,
 ('K', 'MainMemory'): 2.055208921432495,
 ('K', 'FastWeightScratchpad'): 0.322990357875824,
 ('K', 'FastGlobalBuffer'): 0.26214399933815,
 ('K', 'FastOutputScratchpad'): 1.9945430755615234,
 ('Q', 'FastMAC'): 10.737418174743652,
 ('Q', 'FastWeightScratchpad'): 0.322990357875824,
 ('Q', 'FastGlobalBuffer'): 0.26214399933815,
 ('Q', 'MainMemory'): 2.055208921432495,
 ('Q', 'FastInputScratchpad'): 0.80530846118927,
 ('Q', 'FastOutputScratchpad'): 1.9945430755615234,
 ('QK', 'FastMAC'): 21.474836349487305,
 ('QK', 'FastOutputScratchpad'): 3.795562505722046,
 ('QK',

In [140]:
%%capture grid_output 
# Grid for simple example

grid = {}
for sub_arch in ['fast', 'slow']:
    for einsum in ['matmul', 'matvec', 'matmulmatvec']:
        grid[(sub_arch, einsum)] = map(
            "eyeriss-submods/"+sub_arch+".yaml",
            "matmulvecmul/"+einsum+".yaml"
        )

In [141]:
grid

In [142]:
for k, v in grid.items():
    print(k, v['latency'])


In [143]:
grid[('fast', 'matmul')]

In [155]:
{
    k: v['latency']
    for k, v in grid.items()
}

{('fast', 'matmul'): np.float32(0.00016384),
 ('fast', 'matvec'): np.float32(2.08e-05),
 ('fast', 'matmulmatvec'): np.float32(2.08e-05),
 ('slow', 'matmul'): np.float32(0.00049152),
 ('slow', 'matvec'): np.float32(2.08e-05),
 ('slow', 'matmulmatvec'): np.float32(2.08e-05)}

In [124]:
# todo we can autofill this more prettily
# maps component to a list of (einsum, start time, latency)
def lat(comp, ein):
    return grid[(comp, ein)]['latency']

schedule = {
    'fast': [('matmul', 0, lat('fast', 'matmul')),
             ('matmulmatvec', max(lat('fast', 'matmul'), lat('slow', 'matvec')), lat('fast', 'matmulmatvec'))
            ],
    'slow': [('matvec', 0, lat('slow', 'matvec'))]
}

In [125]:
schedule

{'fast': [('matmul', 0, np.float32(0.00016384)),
  ('matmulmatvec', np.float32(0.00016384), np.float32(2.08e-05))],
 'slow': [('matvec', 0, np.float32(2.08e-05))]}

In [126]:
# total latency function: 
# check that the shedule is valid, then sum last start time and last latency for each componenet and take max
overall_lat = max(lat('fast', 'matmul'), lat('slow', 'matvec')) + lat('fast', 'matmulmatvec')
overall_lat

np.float32(0.00018464)

In [127]:
def mem(comp, ein):
    return grid[(comp, ein)]['total r+w']

einsums = [('fast', 'matmul'), ('slow', 'matvec'), ('fast', 'matmulmatvec')]

mems = sum(mem(c, e) for (c, e) in einsums)
bwu = (mems/overall_lat) / 6.4e9, # todo ugly i am hardcoded
bwu

659456.0

In [ ]:
v_slow = map(
    "eyeriss-submods/slow.yaml",
    "gpt_v.yaml"
)

In [ ]:
full_mapping = gpt_full
# Verify that V can be computed in the time that Q, K, QK, QK_softmax are done
full_latencies = full_mapping['mapping'].latency(per_einsum=True)
qk_lat = sum(full_latencies[k] for k in ['Q', 'K', 'QK', 'QK_softmax'])
print("Q K QK QK_soft fast latency", qk_lat)
print("V slow latency", v_slow['mapping'].latency())
# Then the total latency is just the overall latency of the full mapping minus V
para_lat = full_mapping['mapping'].latency() - full_latencies['V']
print("Latency with parallel", para_lat)

actions = full_mapping['mapping'].actions()
rw = actions[('MainMemory', 'read')] + actions[('MainMemory', 'write')]
bwu = (rw/para_lat) / 6.4e9, # todo ugly i am hardcoded
bwu